quantum-cortex — session B re read on a mounted checkpoint, with every answer kept per line (Rev66)

Settings: Accelerator = None (CPU is enough, minutes), Internet = On. Secrets: `GITHUB_TOKEN`, `QUANTUM_CORTEX_JOURNAL_KEY` (the same key that sealed the run's journal). Input: the fine tune notebook's version whose Output holds `runs-keep/journal-n1-<V>/` (`ckpt.pt`, `journal.jsonl`, the payload store). Set `V` below. The cell reopens the sealed journal from the disk in a new process and re runs session B on the checkpoint: the numbers must equal the run's committed session B file (checked and printed); the file adds `answers` (entity, schema, the attribute asked, the answer, its outcome, the own line's mark, the cited line's mark, whether the cited line is the own one and the highest marked one) and, from now on, the marks and the oracle. Writes `metrics/mqar/hm-lm-<run_id>-session-b-answers.json`. Download the log as a file before reading any number.


In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # the repository is private
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
# the runs directory lives OUTSIDE the clone (/kaggle/working/runs-keep) and is never deleted: a session
# interruption that re-runs the notebook finds the last checkpoint there and resumes instead of restarting
import os
os.makedirs('/kaggle/working/runs-keep', exist_ok=True)
if not os.path.islink('runs'): os.symlink('/kaggle/working/runs-keep', 'runs')
print('runs ->', os.path.realpath('runs'), '| kept from a previous session:', os.listdir('runs'))
!pip -q install jsonschema datasets cryptography


In [ ]:
# the key: from the Kaggle secret to the environment, never to a file
import os
from kaggle_secrets import UserSecretsClient
os.environ['QUANTUM_CORTEX_JOURNAL_KEY'] = UserSecretsClient().get_secret('QUANTUM_CORTEX_JOURNAL_KEY')
assert len(bytes.fromhex(os.environ['QUANTUM_CORTEX_JOURNAL_KEY'])) == 32, 'the key must be 32 bytes hex'
print('journal key: present (not printed)')
# the checkpoint relay (cortex_data/relay.py): with these two secrets the run pushes its directory to a private
# Kaggle dataset after every checkpoint and pulls it back before a resume, so a relaunch in a fresh container
# restarts from the last thousand steps; without them everything runs as before
try:
    os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret('KAGGLE_KEY')
    from cortex_data import relay
    print('relay:', 'on' if relay.available() else 'off (kaggle command missing)')
except Exception as e:
    print('relay: off (secrets KAGGLE_USERNAME / KAGGLE_KEY not attached)')


In [ ]:
V = 'v9'                                   # the run to re read: v6, v7, v9, ...
CONFIGS = {'v6': 'configs/kaggle_t4_journal_v6.json', 'v7': 'configs/kaggle_t4_journal_v7.json', 'v8': 'configs/kaggle_t4_journal_v8.json', 'v9': 'configs/kaggle_t4_journal_v9.json'}
import glob, os, shutil, json, torch
cands = sorted(glob.glob(f'/kaggle/input/**/journal-n1-{V}/ckpt.pt', recursive=True))
assert cands, f'no Input holds journal-n1-{V}/ckpt.pt: mount the fine tune notebook version of that run'
src = os.path.dirname(cands[0]); dst = f'runs-keep/journal-n1-{V}'
os.makedirs(dst, exist_ok=True)
for f in os.listdir(src):
    s, d = os.path.join(src, f), os.path.join(dst, f)
    if not os.path.exists(d):
        (shutil.copytree if os.path.isdir(s) else shutil.copy)(s, d)
run_id = torch.load(os.path.join(dst, 'ckpt.pt'), map_location='cpu', weights_only=False)['run_id']
out = f'metrics/mqar/hm-lm-{run_id}-session-b-answers.json'
print('run', run_id, '->', out)
!python -m cortex_c2b.hm_lm --session-b --ckpt {dst}/ckpt.pt --journal {dst}/journal.jsonl --config {CONFIGS[V]} --out {out}
new = json.load(open(out)); old_path = f'metrics/mqar/hm-lm-{run_id}-session-b.json'
if os.path.exists(old_path):
    old = json.load(open(old_path))
    keys = [k for k in old if k.startswith('hm_') and not isinstance(old[k], dict)]
    same = {k: (old[k] == new.get(k)) for k in keys}
    print('equals the committed session B on', sum(same.values()), 'of', len(same), 'scalar fields; differing:', [k for k, v in same.items() if not v])
a = new['answers']['on']
print('answers on:', len(a), '| outcomes:', {o: sum(1 for r in a if r['outcome'] == o) for o in ('strict', 'valid', 'invalid', 'abstain')})
inv = [r for r in a if r['outcome'] == 'invalid']
print('invalid citations:', len(inv), '| cite the own line:', sum(1 for r in inv if r.get('cited_pointer_is_own')), '| cite the highest marked line:', sum(1 for r in inv if r.get('cited_is_highest_mark')), '| own line absent:', sum(1 for r in inv if r.get('own_in_window') is False))
ab = [r for r in a if r['outcome'] == 'abstain']
print('false abstentions:', len(ab), '| own line absent from the window:', sum(1 for r in ab if r.get('own_in_window') is False), '| own mark when present:', sorted(round(r['mark_own'], 3) for r in ab if r.get('mark_own') is not None))
neg = [r for r in new['answers']['negctrl'] if r['outcome'] != 'abstain']
print('claims on never planted entities:', len(neg), '| their highest mark:', sorted(round(r['mark_max'], 3) for r in neg if r.get('mark_max') is not None))
